In [ ]:
# Transformer 모델 구축 - Transformer RAG(Retriever Augmentation Generation) 구성 및 모델 파이프라인 구축(데이터 수집기 + Qdrant 의미 기반 검색엔진 적용)
# - Retrieval 단계: Qdrant 의미 기반 검색엔진을 통해 관련 문서를 빠르게 찾아냄
# - Augmentation 단계: 검색된 문서를 기반으로 LLM 입력 프롬프트를 강화 → 맥락 있는 답변 생성
# - Generation 단계: LLM이 최종 답변을 생성 → 단순 생성보다 정확성과 신뢰성이 높아짐
# - 예시) 구글, 네이버에서 사용되는 RAG 시스템 구현

# 학습 목표 - 실무에서 사용되는 파이프라인 이해 및 적용
# 1. 데이터 수집기
# - 수집 대상 도메인: Google News RSS
# - RSS 피드 파싱: feedparser 라이브러리로 RSS XML을 읽고 기사 항목 추출
# - 데이터 정제: title,link,description,pubDate,soure 필드 추출, pubDate -> Python datetime 변화
# - DB 저장: PostgreSQL 테이블 news_articles 에 Insert
# - 수집 데이터 -> PostgreSQL 테이블 생성 및 입력
# - DB 조회 + 로깅 설정으로 데이터 관리

# 2. Qdrant 의미 기반 검색 구축
# - Qdrant 구축 + 뉴스 컬렉션 생성
# - Qdrant 검색엔진 서버 실행(qdrant.exe): .\LLM\qdrant\qdrant.exe
# - https://github.com/qdrant/qdrant/releases 공식사이트: qdrant-x86_64-pc-windows-msvc.zip, 압축해제 qdrant.exe 실행
# - API(curl) 테스트: http://localhost:6333/collections, curl http://localhost:6333/collections

# 3. 임베딩 생성 후 Qdrant Collection에 Insert/Update
# - 임베딩 모델 로드: 온라인, 오프라인 사용
# - 컬렉션에 데이터 삽입: Batch 단위로 Qdrant의 update/insert

# 4. Qdrant 의미 기반 검색
# - 의미 기반 검색으로 관련 문서 조회

# 5. QA 처리
# - Qdrant 검색 결과 -> QA 토크나이저 + QA 모델
# - 형태소 분석으로 한국어 처리 강화
# - 문자열 -> 토큰 ID 변환, 예시: "AI는 의료 분야에서 활용된다." -> [101, 1234, 5678, ...]
# - 모델 입력: input_ids 토큰 ID 배열, attention_mask 패딩 여부 표시, 모델 출력을 텍스트로 복원 [101, 1234, 5678, ...] -> "AI는 의료 분야에서 활용된다."

# 6. 요약 처리
# - 요약 토크나이저 + 요약 모델 (KoBART 기반)
# - 반복 억제, 길이 조절, 다양성 확보 등 파라미터 튜닝
# - 후처리(clean_summary)로 중복 제거
# - from transformers import BartTokenizer # 영어 전용이라 맞지 않음

# 7. 최종 응답: 외부 서비스 연계 검토
# - Local LLM은 GPU 장비 한계로 로컬 실행은 패스 -> 대신 Copilot에 문의하여 자연스러운 답변 재구성
# - 검토: 현재 로컬 GPU 장비로는 생성형 LLM 모델을 도메인에 맞추어 파인튜닝은 불가능, 현재 추론 모델도 좋은 성능의 모델로 교체 불가능

# 8. 서비스: 구글, 네이버 RAG 시스템 구성
# - /llm_app/transformer_rag2_23_app.py
# - FastAPI 구동 정보: 터미널에서 구동, uvicorn transformer_rag2_23_app:app --reload, 경로 포함 uvicorn LLM.llm_app.transformer_rag2_23_app:app --reload
# - FastAPI 서비스: /search, 입력: 질의문, 출력: QA + 요약 결과 + 출처 정보
# - 1)  [사용자 질의]
# - 2)  [FastAPI 엔드포인트: /search]
# - 3)  [SentenceTransformer: 임베딩]
# - 4)  [Qdrant 의미 기반 검색]
# - 5)  [검색 결과 문서]
# - 6)  [KoELECTRA QA 모델 + MeCab 후처리(형태소 분석: 한국어 처리 강황)]
# - 7)  [KoBART Summarization 모델 + clean_summary]
# - 8)  [응답 + 출처 표시]
# - 9)  [최종 응답 LLM 모델은  외부 서비스 연계 검토: 자연스러운 문장]
# - 10) [최종 사용자 응답]

In [12]:
# 데이터 수집기: Google News RSS
import feedparser
from bs4 import BeautifulSoup
from datetime import datetime
import psycopg2
import logging

# 로깅 설정: 파일 저장
logging.basicConfig(
    level=logging.INFO, # 로그 레벨: INFO 이상 기록
    format="%(asctime)s [%(levelname)s] %(message)s", # 로그 출력 형식
    handlers=[
        logging.FileHandler("24.transformer_rag3_app.log", encoding="utf-8"), # 로그를 파일에 저장
        logging.StreamHandler() # 콘솔에도 출력
    ]
)

# DB 연결 한번만 수행
conn = psycopg2.connect(
    dbname="newsdb",      # 생성한 DB 이름
    user="newsuser",      # 생성한 사용자
    password="1234",      # 설정한 비밀번호
    host="localhost",     # 로컬에서 실행 중
    port="5432"           # 기본 포트
)
cur = conn.cursor()

# Google News RSS -> DB Insert
def insert_article(title, content, url, published_at, source_name, source_url):
    try: # 중복 뉴스는 무시되고, 새로운 뉴스만 저장: ON CONFLICT (url) url 컬럼 값이 이미 존재하면 충동 발생, DO NOTHING 충돌시 아무것도 하지 않고 넘어감
        cur.execute("""
            INSERT INTO news_articles (title, content, url, published_at, source_name, source_url)
            VALUES (%s, %s, %s, %s, %s, %s)
            ON CONFLICT (url) DO NOTHING;
        """, (title, content, url, published_at, source_name, source_url))
        if cur.rowcount > 0: # 새로운 행 추가시 1
            logging.info("데이터 Insert 성공: %s", title)
        else: # 새로운 행 추가 없을시 0
            logging.info("중복으로 건너뜀: %s", title)
        conn.commit()
    except Exception as e:
        logging.error("Insert 실패: %s, 에러: %s", title, e)

# Google News RSS
rss_url = "https://news.google.com/rss?hl=ko&gl=KR&ceid=KR:ko"
feed = feedparser.parse(rss_url)

for entry in feed.entries:
    title = entry.title # title
    
    # description HTML 제거
    content_raw = entry.description if "description" in entry else ""
    content = BeautifulSoup(content_raw, "html.parser").get_text()

    url = entry.link # url
    # pubDate 처리
    if hasattr(entry, "published_parsed"): # hasattr() 객체에 안전한 속성 접근을 위해 사용
        published_at = datetime(*entry.published_parsed[:6]) # feedparser가 RSS <pubDate> 태그를 자동으로 읽어와서 속성으로 제공
    else:
        published_at = datetime.now()
    # source 처리
    if hasattr(entry, "source"):
        source_name = entry.source.title # 언론사 이름(본문 텍스트)
        source_url = entry.source.href # 언론사 URL(속성)
    else:
        source_name = "Google News"
        source_url = ""

    # DB Insert 함수
    insert_article(title, content, url, published_at, source_name, source_url)
    # print(f"{title}\n{content}\n{url}\n{published_at}\n{source_name}\n{source_url}") # 데이터 확인

# 마지막 연결 객체 닫기: 순서 지켜서 닫아야 함
cur.close()
conn.close()
logging.info("DB 연결 종료 완료")

2026-04-03 15:26:08,080 [INFO] 중복으로 건너뜀: “이란 석기시대로” 트럼프, 대형 교량 폭격…“다음은 발전소” - 한겨레
2026-04-03 15:26:08,094 [INFO] 중복으로 건너뜀: 李 “韓·佛, 호르무즈 수송로 확보 협력...원자력 협력도 확대" - 조선일보
2026-04-03 15:26:08,098 [INFO] 중복으로 건너뜀: 폭력사위서 딸 지키려고…'캐리어 시신' 장모, 원룸 동거 택했다 - 한국경제
2026-04-03 15:26:08,102 [INFO] 데이터 Insert 성공: 민주당 지도부, ‘돈봉투 의혹 제명 불복’ 김관영에 “국민 눈높이 안 맞아” 선 긋기 - 경향신문
2026-04-03 15:26:08,108 [INFO] 중복으로 건너뜀: 유엔 안보리서 ‘호르무즈 무력 개방’ 결의안 표결…중·러·프 반대 입장 - 경향신문
2026-04-03 15:26:08,114 [INFO] 데이터 Insert 성공: 민주당, 대구시장 후보 김부겸 만장일치로 단수 공천 - 한겨레
2026-04-03 15:26:08,114 [INFO] 중복으로 건너뜀: 민주 48%·국힘 18%…지지율 격차 30%P까지 벌어졌다 - 동아일보
2026-04-03 15:26:08,125 [INFO] 중복으로 건너뜀: ‘낙화주의보’…밤부터 전국에 비, 제주·남해안엔 강풍 - 한겨레
2026-04-03 15:26:08,130 [INFO] 중복으로 건너뜀: [속보]노무현 사위 곽상언 “유튜버의 민주당 선거 개입, 알만한 사람 다 알아” - 문화일보
2026-04-03 15:26:08,131 [INFO] 중복으로 건너뜀: 1764, 좌익 무장대가 짓밟은 4·3 희생자 숫자입니다 - 조선일보
2026-04-03 15:26:08,140 [INFO] 중복으로 건너뜀: [속보] “이란 ‘중부상공서 미 F-35 격추’” - 매일경제
2026-04-03 15:26:08,145 [INFO] 중복으로 건너뜀: CNN "사우디 미군기지 2천억 사드레이더 이란 공격에 

In [13]:
# 데이터 수집 및 저장
# - 수집 데이터 -> PostgreSQL 테이블 생성 및 입력
# - DB 조회 + 로깅 설정으로 데이터 관리
import psycopg2
import json
import logging

# 로깅 설정: 파일 저장
logging.basicConfig(
    level=logging.INFO, # 로그 레벨: INFO 이상 기록
    format="%(asctime)s [%(levelname)s] %(message)s", # 로그 출력 형식
    handlers=[
        logging.FileHandler("24.transformer_rag3_app.log", encoding="utf-8"), # 로그를 파일에 저장
        logging.StreamHandler() # 콘솔에도 출력
    ]
)
# DB에서 수집데이터 조회
def fetch_data():
    try:
        with psycopg2.connect(
            dbname="newsdb",      # 생성한 DB 이름
            user="newsuser",      # 생성한 사용자
            password="1234",      # 설정한 비밀번호
            host="localhost",     # 로컬에서 실행 중
            port="5432"           # 기본 포트
        ) as conn:
            with conn.cursor() as cur:
                # 모든 컬럼 조회
                cur.execute("SELECT id, title, content, url, published_at, source_name, source_url FROM news_articles;") # SQL 쿼리 실행
                rows = cur.fetchall() # fetchall() 전체 결과를 가져옴

                # 컬럼 이름 가져오기
                col_names = [ desc[0] for desc in cur.description ]

                # 결과를 JSON 형태로 변환
                result = []
                for row in rows:
                    row_dict = dict(zip(col_names, row)) # 결과를 dict 형태로 변환: 컬럼 이름과 데이터 매핑 처리
                    row_dict["published_at"] = str(row_dict["published_at"]) # date 컬럽 -> 문자열 변환
                    result.append(row_dict)
                logging.info("PostgreSQL 연결 및 조회 성공!")
                return result
    except Exception as e:
        logging.error("데이터 조회 중 에러 발생: %s", e)
        return [] # 에러가 발생해도 함수가 항상 빈 리스트를 반환, 프로그램이 멈추지 않고 계속 진행 가능

# DB에서 수집데이터 조회 실행
db_data = fetch_data()
# JSON 변환, ensure_ascii=False 한글 깨짐 방지, indent=4 JSON을 4칸 들여쓰기
print(json.dumps(db_data, ensure_ascii=False, indent=4))

2026-04-03 15:26:21,171 [INFO] PostgreSQL 연결 및 조회 성공!


[
    {
        "id": 1,
        "title": "트럼프 “이란, 무고한 카타르 또 공격하면 가스전 날려버릴 것” 경고 - 조선일보",
        "content": "트럼프 “이란, 무고한 카타르 또 공격하면 가스전 날려버릴 것” 경고  조선일보트럼프 “이란이 추가 공격 안 하면, 이스라엘도 중단”…확전 억제 메시지  한겨레이란 최대 가스전 피격…이란 보복에 국제유가 급등  KBS 뉴스브렌트유 111달러까지 올라…원·달러 환율 1500.7원 마감  지디넷코리아에너지주 급등 불렀지만…가스 공급망 훼손에 연료비 우려  이투데이",
        "url": "https://news.google.com/rss/articles/CBMingFBVV95cUxPa3Z4RmRyOWp5amhXX2dpbVJKOG40TnNYQTF6QkdsNm4yaXVrMldLak9USDlCR2NzeU1ERHdSTnpfYzNFWUhxcVpMOHZQUGpVdzBTMy1ta3FUbi1OaFpIVUYzY1o2VF84YlR3TllkNldRbC02bWtVcUVpclFDMTJuVXBoVDBuN3JSRFg2YzF5Y0RmcXNsU0cwYzd1Vkpxd9IBsgFBVV95cUxOeU90Zi10VWhrOFJMQ21yZF96ODRNRWt4cWRqTjk2LVR0N2E5QWVxcVpRd01EV0JIY0VVdXdZdjhOcXdvdVFFLV9LMExINkVuU0FiVlV3YkRxUTFOSDVsTWN4YUtQcmZtVEU4b19EQTRYODZobGpVT2JEUTFRR0d3QnhCNm9pcWtWb2NqWEQ0QjNIeDFuNEtPN2ZwZ2UxcFdLai1MMHM5STNIdzA1M1pJUFN3?oc=5",
        "published_at": "2026-03-19 04:23:00",
        "source_name": "조선일보",
        "source_url": "https://www.chosun.com"
    },
    {
        "id": 2,
        

In [14]:
# Qdrant 의미 기반 검색 구축
# - Qdrant 구축 + Google News RSS 뉴스 컬렉션 생성
# - Qdrant 검색엔진 서버 실행(qdrant.exe): .\LLM\qdrant\qdrant.exe
# - https://github.com/qdrant/qdrant/releases 공식사이트: qdrant-x86_64-pc-windows-msvc.zip, 압축해제 qdrant.exe 실행
# - API(curl) 테스트: http://localhost:6333/collections, curl http://localhost:6333/collections

import torch
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.http import models # Qdrant 컬렉션 및 검색 관련 설정 정의하는 데이터 모델(구조체 클래스) 로드
# Qdrant 클라이언트 연결: Qdrant 검색엔진 서버 실행 및 서버 연결 확인
qdrant = QdrantClient(host="localhost", port=6333)

# Qdrant 컬렉션 생성
# - 벡터 크기는 모델에 따라 다르다(sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 -> 384차원)
# - 거리(metric)는 보통 COSINE 사용
# 컬렉션 존재 여부 확인 후 생성
if not qdrant.collection_exists("news_articles_collection"):
    qdrant.create_collection(
        collection_name="news_articles_collection",
        vectors_config=models.VectorParams(
            size=384, # 임베딩 벡터 차원: paraphrase-multilingual-MiniLM-L12-v2 임베딩 모델과 차원을 동일하게 맞추어야 한다
            distance=models.Distance.COSINE # 유사도 계산 방식
        )
    )
    print("Qdrant 컬렉션 생성 완료")
else:
    print("news_articles_collection 컬렉션이 존재합니다.")

2026-04-03 15:26:27,972 [INFO] HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
2026-04-03 15:26:30,017 [INFO] HTTP Request: GET http://localhost:6333/collections/news_articles_collection/exists "HTTP/1.1 200 OK"


news_articles_collection 컬렉션이 존재합니다.


In [ ]:
# 임베딩 생성 후 Qdrant Collection에 Insert/Update
# - 임베딩 모델 로드: 온라인, 오프라인 사용
# - 컬렉션에 데이터 삽입: Batch 단위로 Qdrant의 update/insert

# 디바이스 설정
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"PyTorch Version: {torch.__version__}, Device: {device}")

# 임베딩 모델 로드
# - 여기서는 paraphrase-multilingual-MiniLM-L12-v2 모델을 사용했는데, 다국어(한국어 포함) 문장 의미를 잘 반영하는 임베딩을 생성한다
# - 문장을 입력하면 의미 공간에서 가까운 벡터로 변환
# embedder = SentenceTransformer( # 온라인 상태(단, 로컬캐시에 저장)
#     "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
#     device=device
# )

# 오프라인 사용 준비: 온라인 환경에서 모델 다운로드
# - git lfs install
# - git clone https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
embedder = SentenceTransformer( # 오프라인 상태(경로 지정에 저장)
    "./offline_models/paraphrase-multilingual-MiniLM-L12-v2",
    device=device
)


# 컬렉션에 데이터 삽입
def insert_to_qdrant(data):
    ids = []
    vectors = []
    payloads = []
    for item in data:
        text = item["title"] + " " + item["content"] # 제목+내용 임베딩 대상 데이터
        embedding = embedder.encode(text).tolist() # 제목+내용 임베딩

        ids.append(item["id"]) # ID 값
        vectors.append(embedding) # 벡터 데이터
        payloads.append(item) # 원본 데이터
    
    # qdrant.upsert()는 삽입(insert)+업데이트(update) 기능을 동시에 수행, 같은 ID가 있으면 덮어쓰고, 없으면 새로 추가한다
    # { - 데이터 저장 형식
    #     "id": 1,
    #     "vector": [0.123, -0.456, 0.789, ...],
    #     "payload": {
    #         "title": "AI 의료 활용",
    #         "content": "AI가 의료 분야에서 활용되는 사례...",
    #         "date": "2026-03-11",
    #         "author": "홍길동"
    #     }
    # }
    qdrant.upsert(
        collection_name="news_articles_collection",
        points=models.Batch(
            ids=ids,
            vectors=vectors,
            payloads=payloads # payloads(JSON) 형태로 저장
        )
    )
    print("데이터 임베딩 및 Qdrant 저장 완료")

# Batch 단위로 Qdrant의 update/insert 테스트
batch_size = 10
for i in range(0, len(db_data), batch_size):
    chunk = db_data[i:i+batch_size] # 2개씩 잘라서 가져옴
    insert_to_qdrant(chunk) # 잘라낸 청크를 Qdrant에 전달

2026-04-03 15:11:23,166 [INFO] Load pretrained SentenceTransformer: ./offline_models/paraphrase-multilingual-MiniLM-L12-v2


PyTorch Version: 2.6.0, Device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ./offline_models/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

2026-04-03 15:13:53,250 [INFO] HTTP Request: PUT http://localhost:6333/collections/news_articles_collection/points?wait=true "HTTP/1.1 200 OK"


데이터 임베딩 및 Qdrant 저장 완료


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-03 15:13:55,391 [INFO] HTTP Request: PUT http://localhost:6333/collections/news_articles_collection/points?wait=true "HTTP/1.1 200 OK"


데이터 임베딩 및 Qdrant 저장 완료


In [11]:
# Qdrant collection 정보 확인: 전체 조회 + 특정 ID 조회
from qdrant_client import QdrantClient

qdrant = QdrantClient(host="localhost", port=6333)

# 컬렉션 정보: 컬렉션 생성시 설정된 정보 및 상태를 보여주는 메타데이터
info = qdrant.get_collection("news_articles_collection")
print(info)

# 전체 데이터 조회: limit 지정
points = qdrant.scroll(
    collection_name="news_articles_collection",
    limit=10 # 최대 10개 반환
)
print(f"전체 데이터 조회: limit 지정\n{points}")

# 특정 ID 데이터 조회
point = qdrant.retrieve(
    collection_name="news_articles_collection",
    ids=[350,360],
    with_payload=True, # payload 정보 포함
    with_vectors=False # vectors 정보 제외
)
print(f"특정 ID 데이터 조회:\n{point}")

2026-04-03 15:25:20,843 [INFO] HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
2026-04-03 15:25:22,952 [INFO] HTTP Request: GET http://localhost:6333/collections/news_articles_collection "HTTP/1.1 200 OK"


status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=125 segments_count=8 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None, prevent_unoptimized=None), wal_config=WalConf

2026-04-03 15:25:25,034 [INFO] HTTP Request: POST http://localhost:6333/collections/news_articles_collection/points/scroll "HTTP/1.1 200 OK"


전체 데이터 조회: limit 지정
([Record(id=1, payload={'id': 1, 'title': '트럼프 “이란, 무고한 카타르 또 공격하면 가스전 날려버릴 것” 경고 - 조선일보', 'content': '트럼프 “이란, 무고한 카타르 또 공격하면 가스전 날려버릴 것” 경고\xa0\xa0조선일보트럼프 “이란이 추가 공격 안 하면, 이스라엘도 중단”…확전 억제 메시지\xa0\xa0한겨레이란 최대 가스전 피격…이란 보복에 국제유가 급등\xa0\xa0KBS 뉴스브렌트유 111달러까지 올라…원·달러 환율 1500.7원 마감\xa0\xa0지디넷코리아에너지주 급등 불렀지만…가스 공급망 훼손에 연료비 우려\xa0\xa0이투데이', 'url': 'https://news.google.com/rss/articles/CBMingFBVV95cUxPa3Z4RmRyOWp5amhXX2dpbVJKOG40TnNYQTF6QkdsNm4yaXVrMldLak9USDlCR2NzeU1ERHdSTnpfYzNFWUhxcVpMOHZQUGpVdzBTMy1ta3FUbi1OaFpIVUYzY1o2VF84YlR3TllkNldRbC02bWtVcUVpclFDMTJuVXBoVDBuN3JSRFg2YzF5Y0RmcXNsU0cwYzd1Vkpxd9IBsgFBVV95cUxOeU90Zi10VWhrOFJMQ21yZF96ODRNRWt4cWRqTjk2LVR0N2E5QWVxcVpRd01EV0JIY0VVdXdZdjhOcXdvdVFFLV9LMExINkVuU0FiVlV3YkRxUTFOSDVsTWN4YUtQcmZtVEU4b19EQTRYODZobGpVT2JEUTFRR0d3QnhCNm9pcWtWb2NqWEQ0QjNIeDFuNEtPN2ZwZ2UxcFdLai1MMHM5STNIdzA1M1pJUFN3?oc=5', 'published_at': '2026-03-19 04:23:00', 'source_name': '조선일보', 'source_url': 'https://www.chosun.com'}, vector=None, shard_key=Non

2026-04-03 15:25:27,096 [INFO] HTTP Request: POST http://localhost:6333/collections/news_articles_collection/points "HTTP/1.1 200 OK"


특정 ID 데이터 조회:
[Record(id=350, payload={'id': 350, 'title': "삼천당제약 '주가 조작' 등 논란에 강경 대응 예고 - MEDI:GATE NEWS", 'content': "삼천당제약 '주가 조작' 등 논란에 강경 대응 예고\xa0\xa0MEDI:GATE NEWS‘15조 확신’ 삼천당 美 계약…거래소, 비독점 판단·파트너사 검증 불가[삼천당제약 ...\xa0\xa0팜이데일리[사설] 코스닥 1위가 사흘 새 반토막, 막장 ‘투기판’이 달리 있나\xa0\xa0조선일보삼천당제약 시총 1위 찍고 급락…박사 1명 R&D '신뢰 흔들'\xa0\xa0데일리팜삼천당제약, 사흘새 주가 반토막 … 블로거 한 명에 날라간 '황제주'\xa0\xa0뉴데일리 경제", 'url': 'https://news.google.com/rss/articles/CBMiV0FVX3lxTFBleDNvWW0xazVoaGdUYUw2VlZyR3RHU2FGT2tabEpTdzE0VXZmam45WWpQZmMtQW4tUzFKaUZOeEp2ZUhLQnZsTVY0TzRabS1MNGhMdmxBdw?oc=5', 'published_at': '2026-04-03 05:24:33', 'source_name': 'MEDI:GATE NEWS', 'source_url': 'https://www.medigatenews.com'}, vector=None, shard_key=None, order_value=None), Record(id=360, payload={'id': 360, 'title': '\'14년 만에 음방\' 쿨 이재훈 "3주만에 10kg 감량" 전성기 외모 복귀(\'고막남친\') - SPOTV NEWS', 'content': '\'14년 만에 음방\' 쿨 이재훈 "3주만에 10kg 감량" 전성기 외모 복귀(\'고막남친\')\xa0\xa0SPOTV NEWS\'성시경도 감탄\' 이재훈, 리즈 시절 완벽 재현... 비결은 3주 10kg 감량?\xa0\xa0v.daum.net이재훈, 14년